In [382]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
pd.options.display.max_columns = None
import sklearn
import scipy
import scipy.stats as stats
from scipy.stats import skew,boxcox_normmax, zscore
from scipy.special import boxcox1p
from sklearn.preprocessing import OneHotEncoder,LabelEncoder,RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer,KNNImputer
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, make_scorer, mean_absolute_error 
from sklearn.model_selection import KFold, RandomizedSearchCV
from mlxtend.regressor import StackingCVRegressor
from multiprocessing import cpu_count
# from lightgbm import LGBMRegressor
import matplotlib.pyplot as plt
import xgboost as xgb
import seaborn as sns
from catboost import CatBoostRegressor

In [383]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, FunctionTransformer
from sklearn.feature_selection import VarianceThreshold


In [384]:
df_train = pd.read_csv('../data/train.csv', index_col ='Id')
df_test = pd.read_csv('../data/test.csv', index_col ='Id')

In [385]:
target = 'SalePrice'

In [386]:
data = pd.concat([df_train,df_test])

In [387]:
## Combine some of the variables together to add value and decrease number of features
data['TotalBath'] = data[['FullBath', 'BsmtFullBath', 'HalfBath', 'BsmtHalfBath']].fillna(0).dot([1, 1, 0.5, 0.5])
data['TotalSF'] = data[['TotalBsmtSF' , '1stFlrSF' , '2ndFlrSF']].fillna(0).dot([1, 1, 1])
data['TotalPorch'] = data[['OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch']].fillna(0).dot([1, 1, 1, 1])

data = data.drop(columns = ['FullBath', 'BsmtFullBath', 'HalfBath', 'BsmtHalfBath', 'TotalBsmtSF' , '1stFlrSF' , '2ndFlrSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch'])

In [388]:
data['PoolQC'] = np.where(data['PoolQC'].isna(), 0, 1)
data['Fence'] = np.where(data['Fence'].isna(), 0, 1)
data['MiscFeature'] = np.where(data['MiscFeature'].isna(), 0, 1)
data['Alley'] = np.where(data['Alley'].isna(), 0, 1)

data['LuxuryFeature'] =  data[['PoolQC', 'Fence', 'MiscFeature', 'Fireplaces', 'Alley']].dot([1, 1, 1, 1, 1])

data = data.drop(columns = ['PoolQC', 'Fence', 'MiscFeature', 'MiscVal', 'Fireplaces', 'FireplaceQu', 'Alley'])

In [389]:
numerical_features = data.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = data.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

In [390]:
numerical_features.remove(target)

In [391]:
data[categorical_features].isna().mean()[data[categorical_features].isna().mean()>0]

MSZoning        0.001370
Utilities       0.000685
Exterior1st     0.000343
Exterior2nd     0.000343
MasVnrType      0.605002
BsmtQual        0.027749
BsmtCond        0.028092
BsmtExposure    0.028092
BsmtFinType1    0.027064
BsmtFinType2    0.027407
Electrical      0.000343
KitchenQual     0.000343
Functional      0.000685
GarageType      0.053786
GarageFinish    0.054471
GarageQual      0.054471
GarageCond      0.054471
SaleType        0.000343
dtype: float64

In [392]:
data[numerical_features].isna().mean()[data[numerical_features].isna().mean()>0]

LotFrontage    0.166495
MasVnrArea     0.007879
BsmtFinSF1     0.000343
BsmtFinSF2     0.000343
BsmtUnfSF      0.000343
GarageYrBlt    0.054471
GarageCars     0.000343
GarageArea     0.000343
dtype: float64

In [393]:
features_impute_with_median = ['LotFrontage', 'MasVnrArea']
features_impute_with_mode = ['MSZoning', 'Utilities', 'Exterior1st', 'Exterior2nd','Electrical', 'KitchenQual', 'Functional','SaleType']
features_impute_with_zero = ['GarageYrBlt', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'GarageCars', 'GarageArea']
feaures_impute_with_none = ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'GarageType', 'GarageFinish', 
                            'GarageQual', 'GarageCond']

In [394]:
data[features_impute_with_zero] = data[features_impute_with_zero].fillna(0)
data[feaures_impute_with_none] = data[feaures_impute_with_none].fillna('None')

In [395]:
for col in features_impute_with_median:
    data[col] = data.groupby('Neighborhood')[col] \
               .transform(lambda grp: grp.fillna(grp.median()))

for col in features_impute_with_mode:
    data[col] = data.groupby('Neighborhood')[col] \
               .transform(lambda grp: grp.fillna(
                    grp.mode().iat[0] if not grp.mode().empty 
                    else data[col].mode().iat[0]
               ))

In [396]:
data

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,Heating,HeatingQC,CentralAir,Electrical,LowQualFinSF,GrLivArea,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,PoolArea,MoSold,YrSold,SaleType,SaleCondition,SalePrice,TotalBath,TotalSF,TotalPorch,LuxuryFeature
Id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,60,RL,65.0,8450,Pave,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,BrkFace,196.0,Gd,TA,PConc,Gd,TA,No,GLQ,706.0,Unf,0.0,150.0,GasA,Ex,Y,SBrkr,0,1710,3,1,Gd,8,Typ,Attchd,2003.0,RFn,2.0,548.0,TA,TA,Y,0,0,2,2008,WD,Normal,208500.0,3.5,2566.0,61,0
2,20,RL,80.0,9600,Pave,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,NaN,0.0,TA,TA,CBlock,Gd,TA,Gd,ALQ,978.0,Unf,0.0,284.0,GasA,Ex,Y,SBrkr,0,1262,3,1,TA,6,Typ,Attchd,1976.0,RFn,2.0,460.0,TA,TA,Y,298,0,5,2007,WD,Normal,181500.0,2.5,2524.0,0,1
3,60,RL,68.0,11250,Pave,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,BrkFace,162.0,Gd,TA,PConc,Gd,TA,Mn,GLQ,486.0,Unf,0.0,434.0,GasA,Ex,Y,SBrkr,0,1786,3,1,Gd,6,Typ,Attchd,2001.0,RFn,2.0,608.0,TA,TA,Y,0,0,9,2008,WD,Normal,223500.0,3.5,2706.0,42,1
4,70,RL,60.0,9550,Pave,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,Wd Sdng,Wd Shng,NaN,0.0,TA,TA,BrkTil,TA,Gd,No,ALQ,216.0,Unf,0.0,540.0,GasA,Gd,Y,SBrkr,0,1717,3,1,Gd,7,Typ,Detchd,1998.0,Unf,3.0,642.0,TA,TA,Y,0,0,2,2006,WD,Abnorml,140000.0,2.0,2473.0,307,1
5,60,RL,84.0,14260,Pave,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,BrkFace,350.0,Gd,TA,PConc,Gd,TA,Av,GLQ,655.0,Unf,0.0,490.0,GasA,Ex,Y,SBrkr,0,2198,4,1,Gd,9,Typ,Attchd,2000.0,RFn,3.0,836.0,TA,TA,Y,192,0,12,2008,WD,Normal,250000.0,3.5,3343.0,84,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2915,160,RM,21.0,1936,Pave,Reg,Lvl,AllPub,Inside,Gtl,MeadowV,Norm,Norm,Twnhs,2Story,4,7,1970,1970,Gable,CompShg,CemntBd,CmentBd,NaN,0.0,TA,TA,CBlock,TA,TA,No,Unf,0.0,Unf,0.0,546.0,GasA,Gd,Y,SBrkr,0,1092,3,1,TA,5,Typ,None,0.0,None,0.0,0.0,None,None,Y,0,0,6,2006,WD,Normal,NaN,1.5,1638.0,0,0
2916,160,RM,21.0,1894,Pave,Reg,Lvl,AllPub,Inside,Gtl,MeadowV,Norm,Norm,TwnhsE,2Story,4,5,1970,1970,Gable,CompShg,CemntBd,CmentBd,NaN,0.0,TA,TA,CBlock,TA,TA,No,Rec,252.0,Unf,0.0,294.0,GasA,TA,Y,SBrkr,0,1092,3,1,TA,6,Typ,CarPort,1970.0,Unf,1.0,286.0,TA,TA,Y,0,0,4,2006,WD,Abnorml,NaN,1.5,1638.0,24,0
2917,20,RL,160.0,20000,Pave,Reg,Lvl,AllPub,Inside,Gtl,Mitchel,Norm,Norm,1Fam,1Story,5,7,1960,1996,Gable,CompShg,VinylSd,VinylSd,NaN,0.0,TA,TA,CBlock,TA,TA,No,ALQ,1224.0,Unf,0.0,0.0,GasA,Ex,Y,SBrkr,0,1224,4,1,TA,7,Typ,Detchd,1960.0,Unf,2.0,576.0,TA,TA,Y,474,0,9,2006,WD,Abnorml,NaN,2.0,2448.0,0,1


In [397]:
qual_features = [x for x in categorical_features if 'Qual' in x and x != 'OverallQual']
cond_features = [x for x in categorical_features if 'Cond' in x and x not in['OverallCond', 'Condition1', 'Condition2', 'SaleCondition']]

In [398]:
[x for x in categorical_features if 'QC' in x]

['HeatingQC']

In [399]:
unique_values = {col: data[col].dropna().unique().tolist() for col in data.columns}
for col, vals in unique_values.items():
    print(f"{col!r} ({len(vals)} uniques): {vals[:10]}{' …' if len(vals)>10 else ''}")

'MSSubClass' (16 uniques): [60, 20, 70, 50, 190, 45, 90, 120, 30, 85] …
'MSZoning' (5 uniques): ['RL', 'RM', 'C (all)', 'FV', 'RH']
'LotFrontage' (130 uniques): [65.0, 80.0, 68.0, 60.0, 84.0, 85.0, 75.0, 51.0, 50.0, 70.0] …
'LotArea' (1951 uniques): [8450, 9600, 11250, 9550, 14260, 14115, 10084, 10382, 6120, 7420] …
'Street' (2 uniques): ['Pave', 'Grvl']
'LotShape' (4 uniques): ['Reg', 'IR1', 'IR2', 'IR3']
'LandContour' (4 uniques): ['Lvl', 'Bnk', 'Low', 'HLS']
'Utilities' (2 uniques): ['AllPub', 'NoSeWa']
'LotConfig' (5 uniques): ['Inside', 'FR2', 'Corner', 'CulDSac', 'FR3']
'LandSlope' (3 uniques): ['Gtl', 'Mod', 'Sev']
'Neighborhood' (25 uniques): ['CollgCr', 'Veenker', 'Crawfor', 'NoRidge', 'Mitchel', 'Somerst', 'NWAmes', 'OldTown', 'BrkSide', 'Sawyer'] …
'Condition1' (9 uniques): ['Norm', 'Feedr', 'PosN', 'Artery', 'RRAe', 'RRNn', 'RRAn', 'PosA', 'RRNe']
'Condition2' (8 uniques): ['Norm', 'Artery', 'RRNn', 'Feedr', 'PosN', 'PosA', 'RRAn', 'RRAe']
'BldgType' (5 uniques): ['1Fam', '

In [400]:
exp_to_ord_mapping = {'No' : 'Po', 'Mn': 'Fa', 'Av': 'TA'}

In [401]:
data['BsmtExposure'] = data['BsmtExposure'].apply(lambda x: exp_to_ord_mapping.get(x, x))

In [402]:
ord_map = {'None': 0, 'Po':1, 'Fa':2, 'TA':3, 'Gd':4, 'Ex':5}

In [403]:
for col in qual_features + cond_features + ['HeatingQC', 'BsmtExposure']:
    data[col] = data[col].map(ord_map).astype(int)

In [404]:
bsmt_ord_mapping = {'GLQ': 6,
                     'ALQ': 5,
                     'BLQ': 4,
                     'Rec': 3,
                     'LwQ': 2,
                     'Unf': 1,
                     'None': 0}

In [405]:
for col in ['BsmtFinType1', 'BsmtFinType2']:
    data[col] = data[col].map(bsmt_ord_mapping).astype(int)

In [406]:
## One-hot encoding

remaining_categorical = data.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
data = pd.get_dummies(data, columns=remaining_categorical, drop_first=True, dtype=int)


In [407]:
data['Mo_sin'] = np.sin(2 * np.pi * data['MoSold'] / 12)
data['Mo_cos'] = np.cos(2 * np.pi * data['MoSold'] / 12)

data = data.drop(columns=['MoSold'])

In [408]:
train_mask = ~data[target].isna()

In [409]:
y = data[[target]]
data = data.drop(columns=[target])

In [410]:
var_thresh = 0.01  # drop columns with variance < 0.01
selector = VarianceThreshold(threshold=var_thresh)
# Fit on train
selector.fit(data.loc[train_mask])

VarianceThreshold(threshold=0.01)

In [411]:
# Keep only high-variance columns
cols_to_keep = data.iloc[:, selector.get_support(indices=True)].columns
data = pd.concat([data[cols_to_keep], y], axis=1)

In [412]:
corr_threshold = 0.05

corrs = data[train_mask].drop(columns=[target]).corrwith(data.loc[train_mask, target]).abs()
good_feats = corrs[corrs >= corr_threshold].index.tolist()

In [413]:
data = data[good_feats + [target]]

In [414]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def calculate_vif(df):
    """
    Calculate VIF for each feature in a DataFrame.
    df: pandas DataFrame of only the numeric features to test.
    Returns a DataFrame with ['feature','VIF'].
    """
    vif_data = []
    for i, col in enumerate(df.columns):
        vif = variance_inflation_factor(df.values, i)
        vif_data.append((col, vif))
    return pd.DataFrame(vif_data, columns=['feature','VIF'])

def iterative_vif(df, thresh=10.0, verbose=True):
    """
    Iteratively drop the feature with the highest VIF above the threshold.
    df: DataFrame of numeric features (no target, no dummies you’re not testing).
    thresh: maximum allowed VIF.
    Returns a DataFrame with only the low-VIF features.
    """
    features = df.columns.tolist()
    while True:
        vif_df = calculate_vif(df[features])
        max_vif = vif_df['VIF'].max()
        if max_vif <= thresh:
            break
        # find and drop the worst offender
        drop_feat = vif_df.sort_values('VIF', ascending=False).iloc[0]['feature']
        if verbose:
            print(f"Dropping '{drop_feat}' (VIF = {max_vif:.2f})")
        features.remove(drop_feat)
    if verbose:
        print("Final VIFs:")
        print(calculate_vif(df[features]))
    return df[features]

In [415]:

numeric_df = data[train_mask].drop(columns=['SalePrice'])  # keep only numeric predictors
low_vif_df = iterative_vif(numeric_df, thresh=10.0)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


Dropping 'GarageFinish_None' (VIF = inf)
Dropping 'YearBuilt' (VIF = 31693.91)
Dropping 'YearRemodAdd' (VIF = 18361.07)
Dropping 'GarageYrBlt' (VIF = 1066.22)
Dropping 'TotalSF' (VIF = 424.15)
Dropping 'GarageCond' (VIF = 284.06)
Dropping 'GarageQual' (VIF = 157.41)
Dropping 'MSZoning_RL' (VIF = 144.95)
Dropping 'ExterQual' (VIF = 117.60)
Dropping 'TotRmsAbvGrd' (VIF = 94.53)
Dropping 'RoofMatl_CompShg' (VIF = 88.37)
Dropping 'BsmtCond' (VIF = 86.91)
Dropping 'OverallQual' (VIF = 85.58)
Dropping 'KitchenQual' (VIF = 68.31)
Dropping 'Heating_GasA' (VIF = 64.96)
Dropping 'KitchenAbvGr' (VIF = 61.52)
Dropping 'BsmtQual' (VIF = 59.44)
Dropping 'GrLivArea' (VIF = 53.01)
Dropping 'SaleCondition_Partial' (VIF = 45.44)
Dropping 'GarageCars' (VIF = 43.91)
Dropping 'Exterior1st_VinylSd' (VIF = 42.90)
Dropping 'Functional_Typ' (VIF = 41.26)
Dropping 'OverallCond' (VIF = 36.98)
Dropping 'HeatingQC' (VIF = 34.98)
Dropping 'RoofStyle_Gable' (VIF = 34.50)
Dropping 'TotalBath' (VIF = 26.33)
Dropping '

In [418]:
low_vif_df

,MSSubClass,LotArea,MasVnrArea,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtUnfSF,WoodDeckSF,PoolArea,TotalPorch,LuxuryFeature,MSZoning_FV,MSZoning_RH,MSZoning_RM,LotShape_IR2,LotShape_Reg,LandContour_HLS,LotConfig_CulDSac,LotConfig_Inside,Neighborhood_BrDale,Neighborhood_BrkSide,Neighborhood_ClearCr,Neighborhood_CollgCr,Neighborhood_Crawfor,Neighborhood_Edwards,Neighborhood_IDOTRR,Neighborhood_MeadowV,Neighborhood_Mitchel,Neighborhood_NAmes,Neighborhood_NoRidge,Neighborhood_NridgHt,Neighborhood_OldTown,Neighborhood_SWISU,Neighborhood_Sawyer,Neighborhood_Somerst,Neighborhood_StoneBr,Neighborhood_Timber,Condition1_Feedr,BldgType_2fmCon,BldgType_Duplex,BldgType_Twnhs,HouseStyle_1Story,HouseStyle_2Story,HouseStyle_SFoyer,RoofStyle_Hip,Exterior1st_HdBoard,Exterior1st_Wd Sdng,Exterior1st_WdShing,Exterior2nd_CmentBd,Exterior2nd_HdBoard,Exterior2nd_MetalSd,Exterior2nd_Plywood,Exterior2nd_VinylSd,Exterior2nd_Wd Sdng,MasVnrType_BrkFace,MasVnrType_Stone,Foundation_CBlock,Foundation_PConc,Foundation_Slab,Electrical_FuseF,Functional_Min1,Functional_Min2,GarageType_BuiltIn,GarageType_Detchd,GarageType_None,GarageFinish_RFn,GarageFinish_Unf,PavedDrive_P,SaleType_New,SaleCondition_Normal,Mo_sin
Id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,60,8450,196.0,1,6,706.0,150.0,0,0,61,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,8.660254e-01
2,20,9600,0.0,4,5,978.0,284.0,298,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,5.000000e-01
3,60,11250,162.0,2,6,486.0,434.0,0,0,42,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,-1.000000e+00
4,70,9550,0.0,1,5,216.0,540.0,0,0,307,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,8.660254e-01
5,60,14260,350.0,3,6,655.0,490.0,192,0,84,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,-2.449294e-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1456,60,7917,0.0,1,1,0.0,953.0,0,0,40,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,-8.660254e-01
1457,20,13175,119.0,1,5,790.0,589.0,349,0,0,3,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,1,0,0,0,0,0,1,0,0,1,8.660254e-01
1458,70,9042,0.0,1,6,275.0,877.0,0,0,60,4,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,5.000000e-01


In [419]:
from scipy.stats import skew

# assume train_df is just the training portion, and num_cols is your list of numeric features
skewness = low_vif_df.apply(lambda col: skew(col.dropna()))
skewed_feats = skewness[abs(skewness) > 0.75].index.tolist()
print("Highly skewed features:", skewed_feats)

Highly skewed features: ['MSSubClass', 'LotArea', 'MasVnrArea', 'BsmtExposure', 'BsmtFinSF1', 'BsmtUnfSF', 'WoodDeckSF', 'PoolArea', 'TotalPorch', 'LuxuryFeature', 'MSZoning_FV', 'MSZoning_RH', 'MSZoning_RM', 'LotShape_IR2', 'LandContour_HLS', 'LotConfig_CulDSac', 'LotConfig_Inside', 'Neighborhood_BrDale', 'Neighborhood_BrkSide', 'Neighborhood_ClearCr', 'Neighborhood_CollgCr', 'Neighborhood_Crawfor', 'Neighborhood_Edwards', 'Neighborhood_IDOTRR', 'Neighborhood_MeadowV', 'Neighborhood_Mitchel', 'Neighborhood_NAmes', 'Neighborhood_NoRidge', 'Neighborhood_NridgHt', 'Neighborhood_OldTown', 'Neighborhood_SWISU', 'Neighborhood_Sawyer', 'Neighborhood_Somerst', 'Neighborhood_StoneBr', 'Neighborhood_Timber', 'Condition1_Feedr', 'BldgType_2fmCon', 'BldgType_Duplex', 'BldgType_Twnhs', 'HouseStyle_2Story', 'HouseStyle_SFoyer', 'RoofStyle_Hip', 'Exterior1st_HdBoard', 'Exterior1st_Wd Sdng', 'Exterior1st_WdShing', 'Exterior2nd_CmentBd', 'Exterior2nd_HdBoard', 'Exterior2nd_MetalSd', 'Exterior2nd_Plywo

In [420]:
for feat in skewed_feats:
    low_vif_df[feat] = np.log1p(low_vif_df[feat])

/var/folders/40/vgskp5ms6k56tfv_n3ct_kkw0000gn/T/ipykernel_2259/1173412312.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  low_vif_df[feat] = np.log1p(low_vif_df[feat])


In [421]:
y

,SalePrice
Id,
1,208500.0
2,181500.0
3,223500.0
4,140000.0
5,250000.0
...,...
2915,NaN
2916,NaN
2917,NaN


In [429]:
X_train = low_vif_df
y_train = y.dropna()

X_test = data[~train_mask]

In [430]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

model = RandomForestRegressor(n_estimators=200, random_state=42)
scores = -cross_val_score(
    model, X_train, y_train,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
print("Baseline RF CV RMSE: {:.2f}".format(scores.mean()))

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklea

Baseline RF CV RMSE: 41624.87


In [432]:
r2_scores = cross_val_score(
    model, X_train, y_train,
    cv=5,
    scoring='r2',
    n_jobs=-1
)
print(f"CV R²: {r2_scores.mean():.3f} ± {r2_scores.std():.3f}")

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklea

CV R²: 0.723 ± 0.031
